# 08_人資_訓練滿意度

缺值與平均數

任務：說明空白與 0 分的差別，列出每題有效回覆數。；用 np.nanmean 算三題平均。；用 np.nanstd 計算母體標準差，說明此處只描述這批回覆。

規則：評分 1～5 分，空白代表未答；不把缺答補 0；標準差使用 ddof=0。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('08_人資_訓練滿意度.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '08_人資_訓練滿意度'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
import numpy as np
a = np.array([[r[k] if r[k] is not None else np.nan for k in ['課程內容','講師表達','實務幫助']] for r in rows])
valid = np.sum(~np.isnan(a), axis=0)
means = np.nanmean(a, axis=0)
print('有效回覆：', valid, '平均：', means, '標準差：', np.nanstd(a,axis=0))
assert valid.tolist() == [4,4,4]
assert np.allclose(means, [4,4,4])
assert np.allclose(np.nanstd(a,axis=0), [0.70710678]*3)
wb.close()

## 驗收答案

三題各有 4 筆有效回答，平均皆為 4.0，母體標準差皆約 0.7071。